In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "catalog")
dbutils.widgets.text("data_source", "customers", "Data_source")


In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-dp3/{data_source}/*.csv'
print(base_path)

In [0]:
df = (
     spark.read.format("csv")
     .option("header", True)
     .option("inferSchema", True)
     .load(base_path)
     .withColumn("read_timestamp", F.current_timestamp())
     .select("*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### Silver Processing


In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
print('Rows before duplicates dropped: ' , df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print('Rows after duplicates dropped: ', df_silver.count())

In [0]:
#check those values
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver.select('city').distinct().show()

In [0]:
#typos -> correct names
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDehli': 'New Dehli',
    'NewDheli': 'New Dehli',
    'NewDehlee': 'New Dehli'
}

allowed = ['Bengaluru', 'Hyderabad', 'New Dehli']

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
        .when(F.col("city").isin(allowed), F.col("city"))
        .otherwise(None)
    )
)

df_silver.select('city').distinct().show()         


In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
    .otherwise(F.initcap("customer_name"))
)

df_silver.select('customer_name').distinct().show()

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Fitfuel Market', 'Athlete Choice Store', 'Endurance Foods', 'Hydroboost Nutrition', 'Zenathlete Foods', 'Peak Performance Store', 'Primefuel Nutrition', 'Recovery Lane', 'Staminax Store', 'Eliteathlete Nutrition', "Champion's Choice" and "Athlete's Choice Store"]
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
# Assumption: assign one of the 3 allowed cities where city is NULL
assumed_city = "Bengaluru"  # one of: Bengaluru, Hyderabad, New Dehli

df_silver = df_silver.withColumn(
    "city",
    F.when(F.col("city").isNull(), F.lit(assumed_city)).otherwise(F.col("city"))
)

# Verify no NULL cities remain
df_silver.filter(F.col("city").isNull()).show(truncate=False)
df_silver.select("customer_id", "customer_name", "city").show(truncate=False)

In [0]:
# df_fix: per-customer city corrections recovered from the bronze source.
# Instead of hardcoding the known source spellings, normalize each bronze city
# value (lowercase, strip spaces/punctuation) and fuzzy-match it to the nearest
# allowed city (Levenshtein distance <= 3). Any remaining typo variant is then
# recovered too, so no customer keeps a NULL city.

allowed_cities = ['Bengaluru', 'Hyderabad', 'New Dehli']
allowed_keys = {c: c.lower().replace(' ', '') for c in allowed_cities}

# normalized key of the raw bronze city value
city_key = F.regexp_replace(F.lower(F.trim(F.col('city'))), '[^a-z]', '')

# distance from the bronze value to each allowed city, plus the closest match
distances = [
    (F.levenshtein(city_key, F.lit(key)), F.lit(city))
    for city, key in allowed_keys.items()
]
min_distance = F.least(*[d for d, _ in distances])

fixed_city_expr = F.coalesce(*[
    F.when(min_distance <= F.lit(3), F.when(d == min_distance, c))
    for d, c in distances
])

df_fix = (
    df_bronze
    .filter(F.col('city').isNotNull())
    .withColumn('fixed_city', fixed_city_expr)
    .filter(F.col('fixed_city').isNotNull())
    .groupBy('customer_id')
    .agg(F.first('fixed_city').alias('fixed_city'))
)

df_silver = (
    df_silver
    .join(df_fix, 'customer_id', 'left')
    # the recovered source value takes precedence; if bronze has no
    # usable city at all (NULL/blank everywhere, or too far from any
    # allowed city), fall back to the assumed city
    .withColumn('city', F.coalesce('fixed_city', 'city', F.lit(assumed_city)))
    .drop('fixed_city')
)

df_silver.show()

# any row still NULL has no usable city anywhere in bronze — this should be empty:
df_silver.filter(F.col('city').isNull()).show(truncate=False)

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

In [0]:
df_silver = (
    df_silver
    #Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn("customer", F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown"))))

    #Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)
display(df_silver.limit(5))

In [0]:
df_silver.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold Processing



In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

# Take required columns only
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()